# Testing coding and testing agents for human eval dataset

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_ollama.chat_models import ChatOllama
from langgraph.graph import StateGraph, START, END
from typing import List, TypedDict, Dict
from ..code_processor import CodeProcessor

# initialize code processor
code_processor = CodeProcessor()

# Initialize the Ollama model
llm = ChatOllama(model="qwen2.5-coder:7b")

# 1. State with separate memory "sub-folders"
class AgentState(TypedDict):
    problem: str # The coding problem from the HumanEval dataset
    solution: str # The solution code generated by the programmer agent
    tests: str # The tests generated by the test designer agent

    # agent_memories is the "shared project folder"
    agent_memories: Dict[str, List]

# System prompts
PROGRAMMER_SYSTEM_PROMPT = "You are an expert software developer."
TEST_DESIGNER_SYSTEM_PROMPT = "You are an expert program tester."

# 2. Agents coded to access ONLY their own memory
def programmer_agent(state: AgentState) -> AgentState:
    """Programmer agent"""    
    # This agent ONLY gets its memory from the 'programmer' key
    programmer_memory = state['agent_memories'].get('programmer', [])
    
    new_prompt = HumanMessage(content=f"Implement this function:\n\n{state['problem']}")
    messages = [SystemMessage(content=PROGRAMMER_SYSTEM_PROMPT)] + programmer_memory + [new_prompt]
    
    response = llm.invoke(messages)
    solution = response.content
    
    state['solution'] = solution
    # It ONLY writes back to its own memory
    state['agent_memories']['programmer'] = programmer_memory + [new_prompt, AIMessage(content=solution)]
    
    return state

def test_designer_agent(state: AgentState) -> AgentState:
    """Test designer agent"""
        
    # This agent ONLY gets its memory from the 'test_designer' key
    test_designer_memory = state['agent_memories'].get('test_designer', [])
    
    # Note: It can access other parts of the state like the 'solution' if needed
    new_prompt = HumanMessage(content=f"Design tests for this solution:\n\n{state['solution']}")
    messages = [SystemMessage(content=TEST_DESIGNER_SYSTEM_PROMPT)] + test_designer_memory + [new_prompt]
    
    response = llm.invoke(messages)
    tests = response.content

    state['tests'] = tests


    # It ONLY writes back to its own memory
    state['agent_memories']['test_designer'] = test_designer_memory + [new_prompt, AIMessage(content=tests)]
    return state

# 3. Build and run the graph
graph = StateGraph(AgentState)
graph.add_node("programmer", programmer_agent)
graph.add_node("test_designer", test_designer_agent)

graph.add_edge(START, "programmer")
graph.add_edge("programmer", "test_designer")
graph.add_edge("test_designer", END)

# Compile the graph
agent_coder = graph.compile()

# Example problem for testing
sample_problem = '''def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """'''

print("🚀 Starting Multi-Agent Code Generation and Testing System")
print("=" * 60)

# Initialize state
initial_state = {
    "problem": sample_problem,
    "solution": "",
    "tests": "",
    "agent_memories": {"programmer": [], "test_designer": []}
}

# Run the multi-agent workflow
result = agent_coder.invoke(initial_state)

print("=" * 60)
print("✅ Multi-Agent Workflow Complete!")
print(f"📝 Final Solution Length: {len(result['solution'])} characters")
print(f"🧪 Final Tests Length: {len(result['tests'])} characters")

🚀 Starting Multi-Agent Code Generation and Testing System
--- 🔧 Programmer Agent ---
--- 🧪 Test Designer Agent ---
{'problem': 'def has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """', 'solution': "Here is a Python solution using brute force algorithm to compare each number pair:\n\n```python\nfrom typing import List\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    n = len(numbers)\n\n    for i in range(n):\n        for j in range(i + 1, n):\n            if abs(numbers[i] - numbers[j]) < threshold:\n                return True\n                \n    return False\n```\n\nThis function has a time complexity of O(N^2), where N is the length of the input list. This solution will work fine f